In [7]:
import pandas as pd
import yaml
from sqlalchemy import create_engine

In [8]:
with open("../configure.yaml", "r") as f:
    config = yaml.safe_load(f)

db=config["database"]
engine=create_engine(f"postgresql://{db['username']}:{db['password']}@{db['host']}:{db['port']}/{db['database_name']}")

tablas=config["tablas"]

for num, chunk in enumerate(
    pd.read_sql_query(
        f"SELECT * FROM {tablas['train']} ORDER BY date",
        engine,
        chunksize=1000000
    )
):
    chunk.to_parquet(
        f"../data/bronze/train{num}.parquet",
        index=False
    )

In [9]:
def carga_parquet():
    table=["stores","oil","holidays_events","transactions"]
    for i in table:
        database=pd.read_sql(f"SELECT * FROM {tablas[i]}",engine)
        database.to_parquet(f"../data/bronze/{i}.parquet")
carga_parquet()